# Lab 2.3 — Cross-Validation & Advanced Evaluation Metrics
### Module 2 | AI/ML Intermediate Workshop — Nutanix Engineering

> **Kernel:** `aiml-venv`

**Two problems this lab solves:**

1. **Unreliable evaluation:** A single 80/20 split gives one noisy estimate. With 460 samples, you could get PR-AUC 0.78 or 0.91 just from split luck — and ship the wrong model.

2. **Misleading metrics:** On a 15% anomaly rate, accuracy is useless. You need metrics that reflect what the ops team actually cares about: *catching real failures* vs *drowning in false alerts*.

**What you will build:**
- Manual K-Fold cross-validation loop (not just `cross_val_score`)
- Full evaluation suite: Precision, Recall, F1, ROC-AUC, PR-AUC, MCC
- PR and ROC curves with confidence bands from cross-validation
- Threshold selector with real ops constraints
- Calibration analysis

> **Instructor Note:** This lab is deliberately metrics-heavy. Push students to translate every number into an ops sentence — "this means X alerts per day" or "this means Y outages missed per week." Numbers without context are useless.

## 📦 Requirements & Troubleshooting

### Required Packages

| Package | Install Name |
|---------|-------------|
| scikit-learn | `scikit-learn` |
| xgboost | `xgboost` |
| scipy | `scipy` |
| pandas | `pandas` |
| numpy | `numpy` |
| matplotlib | `matplotlib` |
| seaborn | `seaborn` |

**Install all at once:**
```bash
pip install scikit-learn xgboost scipy pandas numpy matplotlib seaborn
```

---

### ⚠️ Common Errors & Fixes

**`ModuleNotFoundError: No module named '...'`**
> Package is missing from the active Python environment.
> Fix: Run the pip install command above in a terminal, then **restart the kernel**.

**`CalledProcessError` — `--break-system-packages` / exit status 2**
> You are using a virtual environment (e.g. `myenv`) where that flag is not supported, or your pip version is old.
> Fix: Open a terminal, activate your venv (`source myenv/bin/activate`), then run `pip install <package>` without that flag.

**`Failed building wheel for <package>` / C extension errors**
> The package does not support your Python version (most common on Python 3.14).
> Fix: Switch the kernel to **Python 3.13**. Click the kernel name in the VS Code top-right corner → *Select Another Kernel* → *Python 3.13*. Then re-run.

**Packages install with no error but `ModuleNotFoundError` still appears**
> You installed into a different Python than the one the notebook is using.
> Fix: Check the kernel shown in the top-right of VS Code. Open a terminal, activate that environment, and install packages there.

**`PermissionError` or `[Errno 13]` when installing**
> Trying to install into a read-only system Python.
> Fix: Use a virtual environment — `python -m venv myenv && source myenv/bin/activate` — then install.


## Environment Check

In [ ]:
import subprocess, sys
required = {'xgboost':'xgboost','sklearn':'scikit-learn','pandas':'pandas',
            'numpy':'numpy','matplotlib':'matplotlib','seaborn':'seaborn','scipy':'scipy'}
for pkg, inst in required.items():
    try: __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install',inst,'--quiet'])
print("All packages ready ✅")

## Imports

In [ ]:
import warnings, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     KFold, TimeSeriesSplit, cross_val_score)
from sklearn.metrics import (
    average_precision_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    precision_recall_curve, roc_curve, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Imports OK ✅')

---
## Step 1: Load Data & Train a Working Model

Load features and train the tuned XGBoost (from Lab 2.2 `best_tuned_model.pkl` if available, else reasonable defaults).

> **Instructor Note:** If students don't have `best_tuned_model.pkl`, the fallback params below are calibrated to give realistic results on the synthetic data.

In [ ]:
import joblib

DATA_PATH = os.path.join('..', 'Module_1', 'features_engineered.csv')
try:
    df = pd.read_csv(DATA_PATH); DATA_SOURCE = 'Module 1 CSV'
except FileNotFoundError:
    np.random.seed(RANDOM_STATE)
    n = 800; cpu = np.random.uniform(5,100,n); resp = np.clip(np.random.exponential(120,n),1,700)
    df = pd.DataFrame({'cpu_percent':cpu,'memory_mb':np.random.uniform(2048,32768,n),
        'disk_io_mbps':np.clip(np.random.exponential(40,n),0,400),'response_time_ms':resp,
        'hour_of_day':np.random.randint(0,24,n),'is_business_hours':np.random.randint(0,2,n),
        'error_rate_per_host':np.random.poisson(2,n),'cpu_rolling_mean_5':cpu+np.random.normal(0,3,n),
        'memory_rolling_mean_5':np.random.uniform(2048,32768,n),'cpu_memory_ratio':cpu/16,
        'io_per_cpu':np.random.exponential(1,n),'log_response_time':np.log1p(resp),
        'host_encoded':np.random.randint(0,8,n),'loglevel_ERROR':np.random.randint(0,2,n),
        'comp_Stargate':np.random.randint(0,2,n),'comp_Cerebro':np.random.randint(0,2,n),
        'is_high_cpu':(cpu>80).astype(int),'is_slow_response':(resp>500).astype(int)})
    DATA_SOURCE = 'Synthetic fallback'

df['is_anomaly'] = df['is_high_cpu'].astype(int)
DROP = [c for c in ['is_high_cpu','is_slow_response','message','date'] if c in df.columns]
df = df.drop(columns=DROP)
feature_cols = [c for c in df.columns if c != 'is_anomaly']
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

X = df[feature_cols]; y = df['is_anomaly']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

# Load tuned model or use good defaults
try:
    model = joblib.load('best_tuned_model.pkl')
    print('Loaded tuned model from Lab 2.2 ✅')
except FileNotFoundError:
    model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=RANDOM_STATE, eval_metric='aucpr', verbosity=0)
    print('Using default params (run Lab 2.2 first for tuned model)')

model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:,1]
y_pred = (y_prob >= 0.5).astype(int)

print(f'Data: {DATA_SOURCE}  |  Shape: {X.shape}  |  Anomaly rate: {y.mean():.1%}')
print(f'Quick check — Test PR-AUC: {average_precision_score(y_test, y_prob):.4f}')

---
## Step 2: The Single-Split Problem

Run the same model with 15 different random train/test splits. Observe how much PR-AUC varies.

> **Instructor Note:** This is the most important motivation for cross-validation. The point lands better as a plot than as an explanation — let students see the variance first, then explain why.

In [ ]:
scores_single = []
for seed in range(15):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed)
    m = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                      random_state=RANDOM_STATE, eval_metric='aucpr', verbosity=0)
    m.fit(X_tr, y_tr)
    scores_single.append(average_precision_score(y_te, m.predict_proba(X_te)[:,1]))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(15), scores_single, color='steelblue', edgecolor='white')
ax.axhline(np.mean(scores_single), color='red', linestyle='--',
           label=f'Mean={np.mean(scores_single):.3f}')
ax.axhline(np.mean(scores_single)+np.std(scores_single), color='orange', linestyle=':',
           label=f'±1 std={np.std(scores_single):.3f}')
ax.axhline(np.mean(scores_single)-np.std(scores_single), color='orange', linestyle=':')
ax.set_xlabel('Random seed'); ax.set_ylabel('PR-AUC')
ax.set_title('PR-AUC Variability Across 15 Different Train/Test Splits')
ax.legend(); plt.tight_layout()
plt.savefig('single_split_variance.png', dpi=120); plt.show()

print(f'PR-AUC range  : [{min(scores_single):.3f}, {max(scores_single):.3f}]')
print(f'Range width   : {max(scores_single)-min(scores_single):.3f}')
print(f'Std deviation : {np.std(scores_single):.3f}')
print(f'\nConclusion: Reporting a single split score is misleading by up to ±{np.std(scores_single):.3f}')

---
## Step 3: Manual K-Fold Cross-Validation

Implement K-Fold manually (without `cross_val_score`) so you can collect per-fold predictions.  
This lets you plot a cross-validated PR curve with confidence bands — something `cross_val_score` hides.

> **Instructor Note:** Always use `StratifiedKFold` for classification — explain that regular KFold may put all anomalies in one fold, making some folds useless for training.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

fold_prauc, fold_f1, fold_rec, fold_prec = [], [], [], []
oof_probs = np.zeros(len(X_train))   # out-of-fold probabilities
oof_true  = np.zeros(len(X_train))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_fold_tr = X_train.iloc[tr_idx]; y_fold_tr = y_train.iloc[tr_idx]
    X_fold_val= X_train.iloc[val_idx]; y_fold_val= y_train.iloc[val_idx]

    fold_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                subsample=0.8, colsample_bytree=0.8,
                                random_state=RANDOM_STATE, eval_metric='aucpr', verbosity=0)
    fold_model.fit(X_fold_tr, y_fold_tr)
    y_prob_val = fold_model.predict_proba(X_fold_val)[:,1]
    y_pred_val = (y_prob_val >= 0.5).astype(int)

    oof_probs[val_idx] = y_prob_val
    oof_true[val_idx]  = y_fold_val

    pa = average_precision_score(y_fold_val, y_prob_val)
    f1 = f1_score(y_fold_val, y_pred_val, zero_division=0)
    rec = recall_score(y_fold_val, y_pred_val, zero_division=0)
    pr  = precision_score(y_fold_val, y_pred_val, zero_division=0)

    fold_prauc.append(pa); fold_f1.append(f1)
    fold_rec.append(rec);  fold_prec.append(pr)
    print(f'  Fold {fold+1}: PR-AUC={pa:.3f}  F1={f1:.3f}  Recall={rec:.3f}  Precision={pr:.3f}')

print(f'\n{"─"*55}')
print(f'  Mean  PR-AUC : {np.mean(fold_prauc):.4f}  ± {np.std(fold_prauc):.4f}')
print(f'  Mean  F1     : {np.mean(fold_f1):.4f}  ± {np.std(fold_f1):.4f}')
print(f'  OOF   PR-AUC : {average_precision_score(oof_true, oof_probs):.4f}  (pooled)')

---
## Step 4: The Full Metrics Suite

Compute every relevant metric and translate each into an ops sentence.

| Metric | Formula | Ops meaning |
|---|---|---|
| **Accuracy** | (TP+TN)/N | Useless at 15% anomaly rate |
| **Precision** | TP/(TP+FP) | Of all alerts fired, what % were real? |
| **Recall** | TP/(TP+FN) | Of all real failures, what % did we catch? |
| **F1** | 2·P·R/(P+R) | Harmonic mean — penalises extreme imbalance |
| **ROC-AUC** | Area under ROC | Ranking quality — optimistic for imbalanced data |
| **PR-AUC** | Area under P-R | Better for imbalanced — baseline = class prevalence |
| **MCC** | Complex | Balanced metric even with class imbalance |

> **Instructor Note:** Ask students to compute "alerts per day" and "failures missed per day" from these numbers, given 2000 daily events and 15% anomaly rate. This makes abstract metrics concrete.

In [ ]:
y_pred_test = (y_prob >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = cm.ravel()

metrics = {
    'Accuracy'  : (tp+tn)/(tp+tn+fp+fn),
    'Precision' : tp/(tp+fp) if (tp+fp)>0 else 0,
    'Recall'    : tp/(tp+fn) if (tp+fn)>0 else 0,
    'F1'        : f1_score(y_test, y_pred_test, zero_division=0),
    'ROC-AUC'   : roc_auc_score(y_test, y_prob),
    'PR-AUC'    : average_precision_score(y_test, y_prob),
    'MCC'       : matthews_corrcoef(y_test, y_pred_test),
    'FPR'       : fp/(fp+tn) if (fp+tn)>0 else 0,
}

print('='*50)
print('FULL EVALUATION REPORT — Test Set')
print('='*50)
for k,v in metrics.items():
    print(f'  {k:<12}: {v:.4f}')

# Ops translation
DAILY_EVENTS  = 2000
ANOMALY_RATE  = y.mean()
daily_anomalies = DAILY_EVENTS * ANOMALY_RATE

print(f'\n── Ops Translation (assuming {DAILY_EVENTS} events/day, {ANOMALY_RATE:.0%} anomaly rate) ──')
print(f'  Real failures per day     : {daily_anomalies:.0f}')
print(f'  Failures caught (Recall)  : {daily_anomalies * metrics["Recall"]:.0f}')
print(f'  Failures MISSED           : {daily_anomalies * (1-metrics["Recall"]):.0f}')
total_alerts = (DAILY_EVENTS * metrics["FPR"] * (1-ANOMALY_RATE)) + daily_anomalies*metrics["Recall"]
false_alerts  = DAILY_EVENTS * metrics["FPR"] * (1-ANOMALY_RATE)
print(f'  Total alerts fired/day    : {total_alerts:.0f}')
print(f'  False alerts/day          : {false_alerts:.0f}')

# Confusion matrix plot
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(cm, display_labels=['Normal','Anomaly']).plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=120); plt.show()

---
## Step 5: PR Curve vs ROC Curve — Why Both Matter

**ROC curve** plots True Positive Rate vs False Positive Rate.  
**PR curve** plots Precision vs Recall.

On imbalanced data, ROC can look deceptively optimistic because the large number of true negatives suppresses the False Positive Rate. PR curve shows the real cost — it directly reflects anomaly detection quality.

> **Instructor Note:** Show both curves side by side. Ask: "Why does the ROC curve look better than the PR curve? Which should you use when reporting to your manager? Which should you use when tuning the model?" 

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ── PR Curve with cross-validated bands ──────────────────────────────────────
mean_recall = np.linspace(0, 1, 100)
pr_curves = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    fm = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                       subsample=0.8, colsample_bytree=0.8,
                       random_state=RANDOM_STATE, eval_metric='aucpr', verbosity=0)
    fm.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    p, r, _ = precision_recall_curve(y_train.iloc[val_idx],
                                      fm.predict_proba(X_train.iloc[val_idx])[:,1])
    pr_interp = np.interp(mean_recall, r[::-1], p[::-1])
    pr_curves.append(pr_interp)
    ax1.plot(r, p, alpha=0.25, color='steelblue', lw=1)

mean_pr = np.mean(pr_curves, axis=0)
std_pr  = np.std(pr_curves, axis=0)
ax1.plot(mean_recall, mean_pr, color='steelblue', lw=2.5,
         label=f'Mean PR-AUC={np.mean(fold_prauc):.3f}')
ax1.fill_between(mean_recall, mean_pr-std_pr, mean_pr+std_pr, alpha=0.2, color='steelblue')
ax1.axhline(y=y.mean(), linestyle='--', color='grey',
            label=f'Random baseline ({y.mean():.3f})')
ax1.set_xlabel('Recall'); ax1.set_ylabel('Precision')
ax1.set_title('Cross-Validated Precision-Recall Curve'); ax1.legend(fontsize=9)

# ── ROC Curve ─────────────────────────────────────────────────────────────────
fpr_test, tpr_test, _ = roc_curve(y_test, y_prob)
ax2.plot(fpr_test, tpr_test, color='darkorange', lw=2,
         label=f'XGBoost (AUC={roc_auc_score(y_test, y_prob):.3f})')
ax2.plot([0,1],[0,1], linestyle='--', color='grey', label='Random (AUC=0.500)')
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve'); ax2.legend(fontsize=9)

plt.suptitle('PR Curve vs ROC Curve — Same Model, Different Stories', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('pr_vs_roc.png', dpi=120); plt.show()

print('Key insight:')
print(f'  ROC-AUC  = {roc_auc_score(y_test, y_prob):.3f}  (looks great)')
print(f'  PR-AUC   = {average_precision_score(y_test, y_prob):.3f}  (harder — this is the real score)')
print(f'  Baseline = {y.mean():.3f}  (random classifier PR-AUC)')

---
## Step 6: Production Threshold Selection

The default threshold of 0.5 was chosen arbitrarily. For Nutanix ops, the right threshold depends on:
- **Recall constraint:** "We must catch at least 80% of real failures"
- **Alert volume constraint:** "The on-call team can handle at most 50 alerts/day"

Find the threshold that satisfies BOTH constraints simultaneously.

> **Instructor Note:** This is real SRE decision-making. Alert fatigue is a documented problem — teams start ignoring alerts when the false alarm rate is too high, which defeats the purpose of monitoring.

In [ ]:
prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_test, y_prob)
f1_arr = 2*prec_arr[:-1]*rec_arr[:-1] / np.maximum(prec_arr[:-1]+rec_arr[:-1], 1e-9)

DAILY_EVENTS    = 2000
RECALL_MIN      = 0.80
MAX_ALERTS_DAY  = 50

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: metrics vs threshold
axes[0].plot(thresh_arr, prec_arr[:-1], label='Precision', color='steelblue', lw=2)
axes[0].plot(thresh_arr, rec_arr[:-1],  label='Recall',    color='darkorange', lw=2)
axes[0].plot(thresh_arr, f1_arr,         label='F1',        color='green', lw=2)
axes[0].axhline(RECALL_MIN, color='red', linestyle=':', label=f'Min recall = {RECALL_MIN}')
axes[0].set_xlabel('Decision Threshold'); axes[0].set_ylabel('Score')
axes[0].set_title('Metrics vs Threshold'); axes[0].legend(fontsize=9)

# Right: alert volume vs threshold
estimated_alerts = []
for thresh in thresh_arr:
    n_alerts = (y_prob >= thresh).sum() / len(y_test) * DAILY_EVENTS
    estimated_alerts.append(n_alerts)

axes[1].plot(thresh_arr, estimated_alerts, color='red', lw=2)
axes[1].axhline(MAX_ALERTS_DAY, color='grey', linestyle='--',
                label=f'Max {MAX_ALERTS_DAY} alerts/day')
axes[1].set_xlabel('Decision Threshold'); axes[1].set_ylabel('Estimated Alerts/Day')
axes[1].set_title('Alert Volume vs Threshold'); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.savefig('threshold_analysis.png', dpi=120); plt.show()

# Find feasible thresholds
alert_arr = np.array(estimated_alerts)
feasible_mask = (rec_arr[:-1] >= RECALL_MIN) & (alert_arr <= MAX_ALERTS_DAY)
feasible_thresholds = thresh_arr[feasible_mask]

if len(feasible_thresholds):
    # Maximise F1 among feasible thresholds
    best_thresh = thresh_arr[feasible_mask][np.argmax(f1_arr[feasible_mask])]
    y_pred_op = (y_prob >= best_thresh).astype(int)
    print(f'✅ Operational threshold found: {best_thresh:.3f}')
    print(f'   Recall    : {recall_score(y_test, y_pred_op):.3f}  (min={RECALL_MIN})')
    print(f'   Precision : {precision_score(y_test, y_pred_op, zero_division=0):.3f}')
    print(f'   F1        : {f1_score(y_test, y_pred_op, zero_division=0):.3f}')
    print(f'   Est. alerts/day: {(y_prob>=best_thresh).mean()*DAILY_EVENTS:.0f}  (max={MAX_ALERTS_DAY})')
else:
    print('⚠️  No threshold satisfies both constraints simultaneously.')
    print('    Options: relax recall threshold, allow more alerts, or improve the model.')

---
## Step 7: Calibration — Do Predicted Probabilities Mean What They Say?

A model that outputs `P=0.9` should be right 90% of the time.  
Tree ensembles are often poorly calibrated — they output extreme values (0.05 or 0.95) more than warranted.

**Why calibration matters for ops:** If you set a threshold of 0.7 to fire alerts, you need to know that "0.7" actually means 70% likely to be an anomaly — not 40%.

> **Instructor Note:** Calibration and discrimination (AUC) are independent properties. A model can have high AUC but poor calibration, or vice versa. Calibration matters when you use the raw probability (e.g., for cost-based thresholding or uncertainty quantification).

In [ ]:
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, label, probs in [(axes[0], 'Before calibration', y_prob)]:
    fraction_pos, mean_pred = calibration_curve(y_test, probs, n_bins=8, strategy='quantile')
    ax.plot(mean_pred, fraction_pos, 's-', color='steelblue', label='Model', lw=2)
    ax.plot([0,1],[0,1], 'k--', label='Perfect calibration')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(label); ax.legend(fontsize=9)

# Calibrate
cal_model = CalibratedClassifierCV(
    XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                  random_state=RANDOM_STATE, eval_metric='aucpr', verbosity=0),
    method='isotonic', cv=3)
cal_model.fit(X_train, y_train)
y_prob_cal = cal_model.predict_proba(X_test)[:,1]

frac_cal, mean_cal = calibration_curve(y_test, y_prob_cal, n_bins=8, strategy='quantile')
axes[1].plot(mean_cal, frac_cal, 's-', color='green', label='Calibrated', lw=2)
axes[1].plot([0,1],[0,1],'k--', label='Perfect')
axes[1].set_xlabel('Mean predicted probability'); axes[1].set_ylabel('Fraction of positives')
axes[1].set_title('After Isotonic Calibration'); axes[1].legend(fontsize=9)

plt.suptitle('Calibration Curves — Are Probabilities Trustworthy?', fontsize=13)
plt.tight_layout(); plt.savefig('calibration_curves.png', dpi=120); plt.show()

print(f'PR-AUC before calibration : {average_precision_score(y_test, y_prob):.4f}')
print(f'PR-AUC after calibration  : {average_precision_score(y_test, y_prob_cal):.4f}')
print('(Calibration may slightly reduce discrimination — that is expected)')

---
## Step 8: Save Evaluation Report

Save all metrics to a JSON file. This is the model's "test certificate" — what it achieved on the locked-out test set, under what conditions.

In [ ]:
report = {
    'model'             : 'XGBClassifier',
    'dataset'           : DATA_SOURCE,
    'n_test_samples'    : int(len(y_test)),
    'anomaly_rate'      : float(y.mean()),
    'cv_pr_auc_mean'    : float(np.mean(fold_prauc)),
    'cv_pr_auc_std'     : float(np.std(fold_prauc)),
    'test_pr_auc'       : float(average_precision_score(y_test, y_prob)),
    'test_roc_auc'      : float(roc_auc_score(y_test, y_prob)),
    'test_f1_at_0.5'    : float(f1_score(y_test, y_pred_test, zero_division=0)),
    'test_recall_at_0.5': float(recall_score(y_test, y_pred_test, zero_division=0)),
    'test_precision_at_0.5': float(precision_score(y_test, y_pred_test, zero_division=0)),
    'test_mcc'          : float(matthews_corrcoef(y_test, y_pred_test)),
}

with open('evaluation_report.json','w') as f: json.dump(report, f, indent=2)

print('Saved: evaluation_report.json')
print()
print(json.dumps(report, indent=2))

---
## Checkpoint — Discussion Questions

1. **You ran the same model 15 times with different splits and got PR-AUC ranging from X to Y. Which number do you report to your manager?**
2. **StratifiedKFold maintains class proportions per fold. Why does this matter specifically for the Nutanix anomaly dataset?**
3. **Your model's ROC-AUC is 0.92 but PR-AUC is 0.61. The class imbalance is 15%. Why the gap? Which metric should drive your threshold decision?**
4. **Calibration reduced PR-AUC by 0.02 but made the probabilities more trustworthy. When is this trade-off worth it? Give a concrete Nutanix use case.**

---
## Key Takeaways

| Technique | When to use |
|---|---|
| StratifiedKFold | Always for classification with imbalanced classes |
| OOF predictions | When you need a full cross-validated PR curve |
| PR-AUC over ROC-AUC | Imbalanced datasets where minority class matters |
| Threshold tuning | Always — translate ops constraints into a threshold |
| Calibration | When raw probabilities feed downstream decisions |

**Next:** Lab 2.4 — SHAP Values & Model Interpretability

---
## 🎯 Assignment — Challenges

### Challenge 1 — Nested Cross-Validation

Standard CV gives optimistic estimates when you've also tuned hyperparameters on the same data (selection bias).  
Nested CV uses an **outer loop** for evaluation and an **inner loop** for tuning.

**Tasks:**
1. Implement nested CV:
   - Outer: `StratifiedKFold(n_splits=5)`
   - Inner: `RandomizedSearchCV(n_iter=15, cv=3)` on XGBoost params
2. Collect outer-fold PR-AUC scores
3. Compare: nested CV PR-AUC vs standard CV PR-AUC from Step 3
4. Answer: Is there a gap? What does a large gap (>0.05) tell you about the tuning process?

*This is expensive (5 × 15 × 3 = 225 model fits) — use `n_jobs=-1`*

In [ ]:
# Challenge 1 — Your solution here



### Challenge 2 — TimeSeriesSplit for Temporal Data

Infrastructure logs are time-ordered. Using random shuffling leaks future information into training.

**Tasks:**
1. Sort the dataset by any time-related column (e.g., `hour_of_day`, or row index as proxy for time)
2. Apply `TimeSeriesSplit(n_splits=5)` — no shuffling
3. Plot the train/validation split windows as a horizontal timeline (use matplotlib `broken_barh`)
4. Compare TimeSeriesSplit CV PR-AUC vs StratifiedKFold CV PR-AUC
5. Answer: Is there a significant difference? What does a higher StratifiedKFold score suggest about the temporal structure of the data?

In [ ]:
# Challenge 2 — Your solution here



### Challenge 3 — Custom Asymmetric Loss Scoring

The ops team says missing a failure costs **8× more** than a false alarm.  
Create a custom scorer that encodes this asymmetry:

`score = recall - (1/8) * false_positive_rate`

**Tasks:**
1. Implement this as a sklearn custom scorer using `make_scorer` and a custom function
2. Use it in a `cross_val_score` call on the XGBoost model
3. Find the decision threshold that **maximises this custom score** (loop over thresholds 0.1–0.9)
4. Compare: what threshold does the custom scorer recommend vs the F1-optimal threshold from Step 6?
5. Plot the custom score vs threshold curve

*`from sklearn.metrics import make_scorer`*

In [ ]:
# Challenge 3 — Your solution here

